In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
BLUE   = "#2563EB";  DBLUE  = "#1D4ED8"
RED    = "#DC2626";  GREEN  = "#16A34A"
ORANGE = "#D97706";  GRAY   = "#6B7280"
TEAL   = "#0891B2";  PURPLE = "#7C3AED"
sns.set_theme(style="whitegrid", font_scale=1.05)

YEAR_COLS = [str(y) for y in range(2015, 2025)]


In [4]:
occ = pd.read_csv(r"D:\Data Analytics\Project 5\Project 5.1\occupazione.csv")        # Employment rates
dis = pd.read_csv(r"D:\Data Analytics\Project 5\Project 5.2\disoccupazione.csv")     # Unemployment rates

for label, df in [("occupazione (Employment)", occ), ("disoccupazione (Unemployment)", dis)]:
    print(f"\n[{label}]")
    print(f"  Shape     : {df.shape}")
    print(f"  AGE groups: {sorted(df['AGE'].unique().tolist())}")
    print(f"  Countries : {len(df['ISO'].unique())} — {sorted(df['ISO'].unique().tolist())}")

common_ages = sorted(set(occ['AGE'].unique()) & set(dis['AGE'].unique()))
print(f"\nCommon age groups (joinable keys): {common_ages}")
print(f"Unemployment-only age groups: {len(set(dis['AGE'].unique()) - set(occ['AGE'].unique()))} extra cohorts")


[occupazione (Employment)]
  Shape     : (420, 13)
  AGE groups: ['15-24', '15-29', '15-64', '20-64', '25-54', '55-64']
  Countries : 35 — ['AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR']

[disoccupazione (Unemployment)]
  Shape     : (2450, 13)
  AGE groups: ['15-19', '15-24', '15-29', '15-39', '15-59', '15-64', '15-74', '20-24', '20-29', '20-64', '25-29', '25-49', '25-54', '25-59', '25-64', '25-74', '30-34', '30-54', '30-64', '30-74', '35-39', '40-44', '40-59', '40-64', '45-49', '50-54', '50-59', '50-64', '50-74', '55-59', '55-64', '60-64', '65-69', '65-74', '70-74']
  Countries : 35 — ['AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR']

Common age groups (joinable ke

In [9]:
# 2. DATA CLEANING & PREPROCESSING
def to_long(df: pd.DataFrame, val_col: str) -> pd.DataFrame:
    """Melt wide format to tidy long format with typed columns."""
    return (df
        .melt(id_vars=['SEX', 'AGE', 'ISO'], value_vars=YEAR_COLS,
              var_name='YEAR', value_name=val_col)
        .assign(YEAR=lambda x: x['YEAR'].astype(int),
                **{val_col: lambda x: pd.to_numeric(x[val_col], errors='coerce')})
    )

occ_l = to_long(occ, 'ERATE')   # Employment Rate
dis_l = to_long(dis, 'URATE')   # Unemployment Rate

In [10]:
# Missing value audit
for label, df_l, val in [("Employment", occ_l, 'ERATE'), ("Unemployment", dis_l, 'URATE')]:
    nan_pct = df_l[val].isna().mean() * 100
    nan_cnt = df_l[val].isna().sum()
    print(f"\n[{label}] Total NaN = {nan_cnt} ({nan_pct:.1f}%)")
    nan_by_iso = df_l[df_l[val].isna()]['ISO'].value_counts()
    if len(nan_by_iso):
        print("  NaN by country:", nan_by_iso.to_dict())


[Employment] Total NaN = 120 (2.9%)
  NaN by country: {'BA': 72, 'ME': 48}

[Unemployment] Total NaN = 3137 (12.8%)
  NaN by country: {'BA': 443, 'ME': 349, 'IS': 249, 'MT': 173, 'NO': 157, 'LU': 149, 'IE': 97, 'LV': 84, 'SK': 84, 'EE': 77, 'BG': 77, 'CY': 75, 'PL': 74, 'RO': 73, 'SI': 71, 'AT': 70, 'LT': 70, 'HR': 62, 'MK': 62, 'HU': 61, 'BE': 60, 'FI': 60, 'PT': 59, 'DE': 54, 'DK': 54, 'CH': 51, 'RS': 50, 'CZ': 44, 'SE': 31, 'FR': 31, 'GR': 20, 'IT': 20, 'ES': 20, 'NL': 18, 'TR': 8}


In [11]:
# Duplicate check
for label, df_l, key in [
    ("Employment", occ_l, ['SEX','AGE','ISO','YEAR']),
    ("Unemployment", dis_l, ['SEX','AGE','ISO','YEAR'])
]:
    dupes = df_l.duplicated(subset=key).sum()
    print(f"\n[{label}] Duplicate rows: {dupes}")


[Employment] Duplicate rows: 0

[Unemployment] Duplicate rows: 0


In [12]:
print("\n[Preprocessing decisions]")
print("  • Wide→Long (melt): Enables time-series groupby and faceting")
print("  • Type casting: YEAR→int64, RATE→float64 (errors='coerce')")
print("  • No imputation: missing values excluded from aggregations")
print("  • BA (Bosnia): data only from 2021; ME (Montenegro): data only to 2021")


[Preprocessing decisions]
  • Wide→Long (melt): Enables time-series groupby and faceting
  • Type casting: YEAR→int64, RATE→float64 (errors='coerce')
  • No imputation: missing values excluded from aggregations
  • BA (Bosnia): data only from 2021; ME (Montenegro): data only to 2021


In [13]:
# Convenience filters — Employment
e_1564 = occ_l[occ_l['AGE'] == '15-64']
e_1524 = occ_l[occ_l['AGE'] == '15-24']
e_5564 = occ_l[occ_l['AGE'] == '55-64']
e_2554 = occ_l[occ_l['AGE'] == '25-54']

In [14]:
# Convenience filters — Unemployment
u_1574 = dis_l[dis_l['AGE'] == '15-74']
u_1524 = dis_l[dis_l['AGE'] == '15-24']
u_5564 = dis_l[dis_l['AGE'] == '55-64']
u_2554 = dis_l[dis_l['AGE'] == '25-54']

In [15]:
# 3. EXPLORATORY DATA ANALYSIS


In [16]:
# ── 3a. Headline trends ──────────────────────────────────────────────────────
emp_yoy = e_1564.groupby('YEAR')['ERATE'].mean()
unemp_yoy = u_1574.groupby('YEAR')['URATE'].mean()

trend_df = pd.DataFrame({
    'Employment Rate (%)': emp_yoy.round(2),
    'Unemployment Rate (%)': unemp_yoy.round(2),
    'ERate YoY pp': emp_yoy.diff().round(2),
    'URate YoY pp': unemp_yoy.diff().round(2),
})
print("\n[Headline Trends 2015–2024 (Employment 15-64 | Unemployment 15-74)]")
print(trend_df.to_string())


[Headline Trends 2015–2024 (Employment 15-64 | Unemployment 15-74)]
      Employment Rate (%)  Unemployment Rate (%)  ERate YoY pp  URate YoY pp
YEAR                                                                        
2015                64.51                  10.33           NaN           NaN
2016                65.57                   9.42          1.06         -0.91
2017                66.89                   8.39          1.32         -1.03
2018                68.07                   7.42          1.18         -0.97
2019                68.88                   6.88          0.81         -0.55
2020                67.70                   7.54         -1.18          0.67
2021                68.77                   7.43          1.08         -0.11
2022                70.46                   6.43          1.69         -1.01
2023                70.88                   6.30          0.42         -0.13
2024                71.34                   6.26          0.46         -0.04


In [17]:
# ── 3b. Employment by age group ──────────────────────────────────────────────
print("\n[Employment stats by age group (2015–2024)]")
print(occ_l.groupby('AGE')['ERATE'].describe().round(2))


[Employment stats by age group (2015–2024)]
       count   mean    std   min    25%    50%    75%   max
AGE                                                        
15-24  680.0  34.67  15.71  10.4  23.48  30.65  45.60  78.2
15-29  680.0  49.35  12.95  23.0  40.05  48.75  57.72  82.5
15-64  680.0  68.31  10.30  29.7  64.00  70.00  75.00  89.4
20-64  680.0  72.78  10.39  32.0  68.90  74.90  79.20  91.1
25-54  680.0  80.18   9.67  35.7  77.50  82.10  86.30  94.8
55-64  680.0  58.20  14.76  16.7  48.72  60.25  69.30  89.7


In [18]:
# ── 3c. Unemployment by age group ───────────────────────────────────────────
print("\n[Unemployment stats by age group — key cohorts (2015–2024)]")
key_ages = ['15-24','25-54','55-64','15-74']
print(dis_l[dis_l['AGE'].isin(key_ages)].groupby('AGE')['URATE'].describe().round(2))


[Unemployment stats by age group — key cohorts (2015–2024)]
       count   mean   std  min   25%   50%    75%   max
AGE                                                    
15-24  679.0  18.32  9.73  4.8  10.9  16.1  22.85  55.0
15-74  680.0   7.64  4.54  1.7   4.6   6.3   9.00  28.9
25-54  680.0   6.94  4.45  1.5   4.1   5.5   8.20  28.7
55-64  633.0   5.98  3.51  1.2   3.6   5.1   7.20  22.6


In [20]:
# ── 3d. Gender analysis ──────────────────────────────────────────────────────
emp_gender = e_1564.groupby(['YEAR','SEX'])['ERATE'].mean().unstack()
emp_gender['gap_MF'] = emp_gender['M'] - emp_gender['F']
print("\n[Employment gender trends (15-64) — M-F gap]")
print(emp_gender.round(2))
unemp_gender = u_1574.groupby(['YEAR','SEX'])['URATE'].mean().unstack()
unemp_gender['gap_FM'] = unemp_gender['F'] - unemp_gender['M']
print("\n[Unemployment gender trends (15-74) — F-M gap]")
print(unemp_gender.round(2))


[Employment gender trends (15-64) — M-F gap]
SEX       F      M  gap_MF
YEAR                      
2015  59.41  69.61   10.20
2016  60.48  70.66   10.18
2017  61.79  71.99   10.19
2018  62.94  73.20   10.26
2019  63.81  73.95   10.14
2020  62.76  72.63    9.87
2021  63.70  73.85   10.15
2022  65.60  75.32    9.73
2023  66.22  75.53    9.31
2024  66.81  75.87    9.05

[Unemployment gender trends (15-74) — F-M gap]
SEX       F      M  gap_FM
YEAR                      
2015  10.49  10.17    0.31
2016   9.57   9.27    0.30
2017   8.63   8.15    0.48
2018   7.68   7.16    0.52
2019   7.18   6.57    0.60
2020   7.72   7.36    0.36
2021   7.75   7.11    0.64
2022   6.73   6.13    0.60
2023   6.53   6.06    0.47
2024   6.45   6.06    0.39


In [21]:
# ── 3e. Country rankings 2024 ────────────────────────────────────────────────
emp_rank = e_1564[e_1564['YEAR']==2024].groupby('ISO')['ERATE'].mean().dropna().sort_values(ascending=False)
unemp_rank = u_1574[u_1574['YEAR']==2024].groupby('ISO')['URATE'].mean().dropna().sort_values()

print("\n[Employment ranking — 2024 (15-64)]")
print(emp_rank.round(2))
print("\n[Unemployment ranking — 2024 (15-74), lowest first]")
print(unemp_rank.round(2))


[Employment ranking — 2024 (15-64)]
ISO
IS    85.20
NL    82.30
CH    80.35
MT    78.50
DE    77.40
DK    77.20
NO    77.00
SE    76.65
EE    75.70
CY    75.70
CZ    75.30
HU    75.05
IE    74.50
AT    74.10
LT    73.60
SI    73.00
PT    72.85
FI    72.60
PL    72.50
SK    72.40
LV    71.15
BG    70.85
LU    69.65
FR    69.00
HR    68.25
BE    66.75
RS    66.25
ES    66.05
RO    63.65
GR    63.30
IT    62.20
MK    57.80
TR    55.05
BA    53.75
Name: ERATE, dtype: float64

[Unemployment ranking — 2024 (15-74), lowest first]
ISO
CZ     2.65
PL     2.90
MT     3.15
DE     3.35
IS     3.55
NL     3.65
SI     3.75
NO     4.00
BG     4.15
IE     4.30
CH     4.40
HU     4.50
CY     4.85
HR     5.00
AT     5.15
SK     5.35
RO     5.40
BE     5.70
DK     6.20
LU     6.45
PT     6.50
IT     6.60
LV     6.90
LT     7.15
FR     7.35
EE     7.55
FI     8.40
SE     8.40
RS     8.65
TR     9.45
GR    10.40
ES    11.45
MK    12.15
BA    13.35
Name: URATE, dtype: float64


In [23]:
# ── 3f. COVID shock ──────────────────────────────────────────────────────────
e_pivot = e_1564.groupby(['ISO','YEAR'])['ERATE'].mean().unstack()
u_pivot = u_1574.groupby(['ISO','YEAR'])['URATE'].mean().unstack()

e_shock = (e_pivot[2020] - e_pivot[2019]).dropna().sort_values()
u_shock = (u_pivot[2020] - u_pivot[2019]).dropna().sort_values(ascending=False)

print("\n[COVID shock — Employment drop 2019→2020 (largest drops)]")
print(e_shock.head(8).round(2))

print("\n[COVID shock — Unemployment rise 2019→2020 (largest rises)]")
print(u_shock.head(8).round(2))


[COVID shock — Employment drop 2019→2020 (largest drops)]
ISO
ME   -5.75
IS   -3.85
IE   -2.90
TR   -2.80
GR   -2.45
ES   -2.40
AT   -1.90
SE   -1.80
dtype: float64

[COVID shock — Unemployment rise 2019→2020 (largest rises)]
ISO
ME    2.75
EE    2.45
LT    2.20
IS    2.00
LV    1.80
SE    1.45
ES    1.40
LU    1.20
dtype: float64


In [24]:
# 4. CORRELATION & RELATIONSHIP ANALYSIS

In [26]:
# Merge employment + unemployment on (ISO, YEAR, SEX) — common age 15-64
merged = pd.merge(
    e_1564.groupby(['ISO','YEAR'])['ERATE'].mean().reset_index(),
    dis_l[dis_l['AGE']=='15-64'].groupby(['ISO','YEAR'])['URATE'].mean().reset_index(),
    on=['ISO','YEAR']
).dropna()
overall_corr = merged['ERATE'].corr(merged['URATE'])
print(f"\nOverall correlation Employment vs Unemployment (15-64, all years): {overall_corr:.3f}")


Overall correlation Employment vs Unemployment (15-64, all years): -0.784


In [27]:
# Per-year correlation
print("\nCorrelation by year:")
for yr in range(2015, 2025):
    sub = merged[merged['YEAR'] == yr]
    if len(sub) > 5:
        r = sub['ERATE'].corr(sub['URATE'])
        print(f"  {yr}: r = {r:.3f}")


Correlation by year:
  2015: r = -0.771
  2016: r = -0.770
  2017: r = -0.780
  2018: r = -0.783
  2019: r = -0.785
  2020: r = -0.767
  2021: r = -0.804
  2022: r = -0.823
  2023: r = -0.786
  2024: r = -0.727


In [28]:
# Youth unemployment vs employment correlation
youth_corr = pd.merge(
    e_1524[e_1524['YEAR']==2024].groupby('ISO')['ERATE'].mean(),
    u_1524[u_1524['YEAR']==2024].groupby('ISO')['URATE'].mean(),
    on='ISO'
).dropna()
youth_r = youth_corr['ERATE'].corr(youth_corr['URATE'])
print(f"\nYouth (15-24) Employment vs Unemployment correlation (2024): {youth_r:.3f}")


Youth (15-24) Employment vs Unemployment correlation (2024): -0.624


In [29]:
# Senior correlation
senior_corr = pd.merge(
    e_5564[e_5564['YEAR']==2024].groupby('ISO')['ERATE'].mean(),
    u_5564[u_5564['YEAR']==2024].groupby('ISO')['URATE'].mean(),
    on='ISO'
).dropna()
senior_r = senior_corr['ERATE'].corr(senior_corr['URATE'])
print(f"Senior (55-64) Employment vs Unemployment correlation (2024): {senior_r:.3f}")

Senior (55-64) Employment vs Unemployment correlation (2024): -0.230


In [30]:
# 5. ADVANCED ANALYSIS

In [31]:
# Youth unemployment improvement 2015→2024
youth_change = (u_pivot_y := dis_l[dis_l['AGE']=='15-24']
    .groupby(['ISO','YEAR'])['URATE'].mean().unstack()
    .assign(change=lambda x: x[2024] - x[2015])
    .dropna(subset=[2015, 2024])['change'])
print("\n[Youth (15-24) unemployment change 2015→2024 — top improvers]")
print(youth_change.sort_values().head(8).round(2))
print("\n[Youth (15-24) unemployment change — most deteriorated]")
print(youth_change.sort_values(ascending=False).head(5).round(2))


[Youth (15-24) unemployment change 2015→2024 — top improvers]
ISO
GR   -27.50
HR   -25.45
ES   -21.75
RS   -21.20
IT   -20.00
CY   -19.90
MK   -17.80
PT   -10.20
Name: change, dtype: float64

[Youth (15-24) unemployment change — most deteriorated]
ISO
EE    4.85
LU    3.95
SE    3.85
DK    2.45
NO    2.15
Name: change, dtype: float64


In [32]:
# Senior unemployment trend
print("\n[Senior (55-64) unemployment trend]")
print(u_5564.groupby('YEAR')['URATE'].mean().round(2))


[Senior (55-64) unemployment trend]
YEAR
2015    8.09
2016    7.44
2017    6.60
2018    5.89
2019    5.36
2020    5.54
2021    5.81
2022    5.13
2023    4.97
2024    4.81
Name: URATE, dtype: float64


In [33]:
# Outlier detection: countries with high unemployment + high employment (paradox?)
sub_2024 = merged[merged['YEAR']==2024].sort_values('URATE', ascending=False)
print("\n[2024 combined snapshot — Employment vs Unemployment (15-64)]")
print(sub_2024.round(2).to_string(index=False))


[2024 combined snapshot — Employment vs Unemployment (15-64)]
ISO  YEAR  ERATE  URATE
 BA  2024  53.75  13.45
 MK  2024  57.80  12.25
 ES  2024  66.05  11.45
 GR  2024  63.30  10.50
 TR  2024  55.05   9.65
 RS  2024  66.25   8.95
 FI  2024  72.60   8.55
 SE  2024  76.65   8.50
 EE  2024  75.70   7.90
 FR  2024  69.00   7.45
 LT  2024  73.60   7.40
 LV  2024  71.15   7.30
 IT  2024  62.20   6.75
 PT  2024  72.85   6.60
 LU  2024  69.65   6.45
 DK  2024  77.20   6.35
 BE  2024  66.75   5.75
 SK  2024  72.40   5.50
 RO  2024  63.65   5.40
 AT  2024  74.10   5.20
 HR  2024  68.25   5.05
 CY  2024  75.70   4.95
 HU  2024  75.05   4.50
 CH  2024  80.35   4.50
 IE  2024  74.50   4.45
 BG  2024  70.85   4.25
 NO  2024  77.00   4.10
 NL  2024  82.30   3.70
 IS  2024  85.20   3.65
 SI  2024  73.00   3.65
 DE  2024  77.40   3.45
 MT  2024  78.50   3.20
 PL  2024  72.50   2.95
 CZ  2024  75.30   2.70


In [34]:
# Recovery speed: time to return to pre-COVID unemployment levels
print("\n[Countries that recovered unemployment below 2019 levels by 2022]")
pre_covid = u_pivot[2019]
post_covid = u_pivot[2022]
recovered = (post_covid < pre_covid).dropna()
print(recovered[recovered == True].index.tolist())

print("\n[Countries still above pre-COVID unemployment in 2022]")
print(recovered[recovered == False].index.tolist())


[Countries that recovered unemployment below 2019 levels by 2022]
['BG', 'CH', 'CY', 'DK', 'ES', 'FR', 'GR', 'IE', 'IT', 'LT', 'LU', 'MK', 'MT', 'NO', 'PL', 'PT', 'RS', 'SI', 'TR']

[Countries still above pre-COVID unemployment in 2022]
['AT', 'BA', 'BE', 'CZ', 'DE', 'EE', 'FI', 'HR', 'HU', 'IS', 'LV', 'ME', 'NL', 'RO', 'SE', 'SK']


In [35]:
# 6. VISUALISATIONS

In [38]:
import os

# Define output directory
OUT_DIR = r"D:\Data Analytics\Project 5\Project 5.2\Output"

# Create folder if it doesn't exist
os.makedirs(OUT_DIR, exist_ok=True)

# ── FIG 1: Dual-axis trend (Employment + Unemployment) ──────────────────────
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(emp_yoy.index, emp_yoy.values, color=BLUE, lw=2.5, marker='o', ms=5,
         label='Employment Rate (15-64)')
ax2.plot(unemp_yoy.index, unemp_yoy.values, color=RED, lw=2.5, marker='s', ms=5,
         label='Unemployment Rate (15-74)')

ax1.axvspan(2019.5, 2020.5, color='gray', alpha=0.15, label='COVID-19')

ax1.set_xlabel("Year")
ax1.set_ylabel("Employment Rate (%)", color=BLUE)
ax2.set_ylabel("Unemployment Rate (%)", color=RED)

ax1.tick_params(axis='y', labelcolor=BLUE)
ax2.tick_params(axis='y', labelcolor=RED)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

ax1.set_title(
    "European Employment & Unemployment Trends — 2015–2024\n(Mean across 35 countries)",
    fontweight='bold', fontsize=13
)

plt.tight_layout()

# Save to specified folder
output_path = os.path.join(OUT_DIR, "fig1_dual_trend.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig1_dual_trend.png


In [39]:
# ── FIG 2: Gender split — Employment & Unemployment ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for sex, color, lbl in [('F', RED, 'Female'), ('M', BLUE, 'Male')]:
    s = e_1564.groupby(['YEAR','SEX'])['ERATE'].mean().unstack()[sex]
    ax.plot(s.index, s.values, color=color, lw=2.5, marker='o', ms=5, label=lbl)
ax.axvspan(2019.5, 2020.5, color='gray', alpha=0.12)
ax.set_title("Employment Rate by Gender (15-64)", fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Rate (%)"); ax.legend()

ax = axes[1]
for sex, color, lbl in [('F', RED, 'Female'), ('M', BLUE, 'Male')]:
    s = u_1574.groupby(['YEAR','SEX'])['URATE'].mean().unstack()[sex]
    ax.plot(s.index, s.values, color=color, lw=2.5, marker='o', ms=5, label=lbl)
ax.axvspan(2019.5, 2020.5, color='gray', alpha=0.12)
ax.set_title("Unemployment Rate by Gender (15-74)", fontweight='bold')
ax.set_xlabel("Year"); ax.set_ylabel("Rate (%)"); ax.legend()

plt.suptitle("Gender Comparison: Employment & Unemployment 2015–2024", fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()

# Save to specified folder
output_path = os.path.join(OUT_DIR,"  ✓ fig2_gender_split.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\  ✓ fig2_gender_split.png


In [42]:
# ── FIG 3: Scatter — Employment vs Unemployment (2024) ──────────────────────
fig, ax = plt.subplots(figsize=(11, 8))
for _, row in merged[merged['YEAR']==2024].iterrows():
    ax.scatter(row['ERATE'], row['URATE'], s=60, color=BLUE, alpha=0.7, zorder=3)
    ax.annotate(row['ISO'], (row['ERATE'], row['URATE']),
                textcoords="offset points", xytext=(4, 3), fontsize=8, color=GRAY)
m, b = np.polyfit(merged[merged['YEAR']==2024]['ERATE'].dropna(),
                  merged[merged['YEAR']==2024]['URATE'].dropna(), 1)
x_line = np.linspace(merged['ERATE'].min(), merged['ERATE'].max(), 100)
ax.plot(x_line, m * x_line + b, color=RED, lw=1.5, linestyle='--', label=f'OLS (r={overall_corr:.2f})')
ax.set_xlabel("Employment Rate % (15-64)"); ax.set_ylabel("Unemployment Rate % (15-64)")
ax.set_title("Employment vs. Unemployment — 2024\n(r = −0.73: strong negative correlation)", fontweight='bold', fontsize=13)
ax.legend(); plt.tight_layout()

output_path = os.path.join(OUT_DIR,"fig3_scatter_correlation.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig3_scatter_correlation.png


In [43]:
# ── FIG 4: Youth unemployment country ranking 2024 ──────────────────────────
yu_2024 = (u_1524[u_1524['YEAR']==2024]
    .groupby('ISO')['URATE'].mean()
    .dropna().sort_values(ascending=False))
colors_bar = [RED if v>=20 else (ORANGE if v>=12 else GREEN) for v in yu_2024.values]
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(yu_2024.index[::-1], yu_2024.values[::-1], color=colors_bar[::-1], edgecolor='white')
ax.axvline(yu_2024.mean(), color='navy', lw=1.5, linestyle='--', label=f'Mean={yu_2024.mean():.1f}%')
ax.set_title("Youth (15-24) Unemployment Rate by Country — 2024\nRed ≥ 20% | Orange ≥ 12% | Green < 12%", fontweight='bold', fontsize=13)
ax.set_xlabel("Unemployment Rate (%)"); ax.legend()
plt.tight_layout()

output_path = os.path.join(OUT_DIR,"fig4_youth_unemployment.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig4_youth_unemployment.png


In [44]:
# ── FIG 5: COVID double shock ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
colors_e = [RED if v < 0 else GREEN for v in e_shock.values]
ax.bar(e_shock.index, e_shock.values, color=colors_e, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Employment Change 2019→2020 (pp)", fontweight='bold')
ax.set_ylabel("pp change"); plt.sca(ax); plt.xticks(rotation=45, ha='right')

ax = axes[1]
colors_u = [RED if v > 0 else GREEN for v in u_shock.values]
ax.bar(u_shock.index, u_shock.values, color=colors_u, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Unemployment Change 2019→2020 (pp)", fontweight='bold')
ax.set_ylabel("pp change"); plt.sca(ax); plt.xticks(rotation=45, ha='right')

plt.suptitle("COVID-19 Labour Market Shock by Country", fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()

output_path = os.path.join(OUT_DIR,"fig5_covid_double_shock.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig5_covid_double_shock.png


In [46]:
# ── FIG 6: Unemployment heatmap ──────────────────────────────────────────────
heat_u = (u_1574.groupby(['ISO','YEAR'])['URATE'].mean().unstack().dropna(how='all')
    .sort_values(2024, ascending=False))
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(heat_u, annot=True, fmt=".1f", cmap="RdYlGn_r",
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Unemployment Rate (%)'})
ax.set_title("Unemployment Rate Heatmap by Country & Year (15-74, avg M+F)\nRed = High Unemployment", fontweight='bold', fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("Country")
plt.tight_layout()

output_path = os.path.join(OUT_DIR,"fig6_unemployment_heatmap.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig6_unemployment_heatmap.png


In [47]:
# ── FIG 7: Age group unemployment trend ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
age_sel = ['15-24', '25-54', '55-64', '15-74']
colors_age = [RED, BLUE, ORANGE, GRAY]
for age, color in zip(age_sel, colors_age):
    s = dis_l[dis_l['AGE']==age].groupby('YEAR')['URATE'].mean()
    ax.plot(s.index, s.values, color=color, lw=2.5, marker='o', ms=4, label=age)
ax.axvspan(2019.5, 2020.5, color='gray', alpha=0.12, label='COVID-19')
ax.set_title("Unemployment Rate by Age Group — 2015–2024", fontweight='bold', fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("Unemployment Rate (%)")
ax.legend(title="Age Group"); plt.tight_layout()


output_path = os.path.join(OUT_DIR,"fig7_unemployment_by_age.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig7_unemployment_by_age.png


In [48]:
# ── FIG 8: Youth unemployment improvement 2015→2024 ─────────────────────────
youth_pivot = dis_l[dis_l['AGE']=='15-24'].groupby(['ISO','YEAR'])['URATE'].mean().unstack()
yu_change = (youth_pivot[2024] - youth_pivot[2015]).dropna().sort_values()
colors_yu = [GREEN if v < 0 else RED for v in yu_change.values]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(yu_change.index, yu_change.values, color=colors_yu, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_title("Youth (15-24) Unemployment Change 2015→2024 by Country (pp)\nGreen = Improved | Red = Deteriorated", fontweight='bold', fontsize=13)
ax.set_ylabel("pp change"); plt.xticks(rotation=45, ha='right')
plt.tight_layout()


output_path = os.path.join(OUT_DIR,"fig8_youth_unemployment_change.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig8_youth_unemployment_change.png


In [49]:
# ── FIG 9: Gender unemployment gap per country 2024 ─────────────────────────
g_unemp = (u_1574[u_1574['YEAR']==2024]
    .groupby(['ISO','SEX'])['URATE'].mean()
    .unstack()
    .dropna()
    .assign(gap=lambda x: (x['F'] - x['M']).round(2))
    .sort_values('gap', ascending=False))
colors_g = [RED if v > 1 else (ORANGE if v > 0 else GREEN) for v in g_unemp['gap'].values]

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(g_unemp.index, g_unemp['gap'].values, color=colors_g, edgecolor='white')
ax.axhline(0, color='black', lw=1)
ax.set_title("Unemployment Gender Gap (F−M) by Country — 2024\nRed = Women more unemployed | Green = Men more unemployed", fontweight='bold', fontsize=13)
ax.set_ylabel("F-M gap (pp)"); plt.xticks(rotation=45, ha='right')
plt.tight_layout()


output_path = os.path.join(OUT_DIR,"fig9_unemployment_gender_gap.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig9_unemployment_gender_gap.png


In [51]:
# ── FIG 10: Employment heatmap ───────────────────────────────────────────────
heat_e = (e_1564.groupby(['ISO','YEAR'])['ERATE'].mean().unstack()
    .dropna(how='all').sort_values(2024, ascending=False))
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(heat_e, annot=True, fmt=".1f", cmap="YlGn",
            linewidths=0.3, ax=ax, cbar_kws={'label': 'Employment Rate (%)'})
ax.set_title("Employment Rate Heatmap by Country & Year (15-64, avg M+F)", fontweight='bold', fontsize=13)
ax.set_xlabel("Year"); ax.set_ylabel("Country")
plt.tight_layout()


output_path = os.path.join(OUT_DIR,"fig10_employment_heatmap.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig10_employment_heatmap.png


In [52]:
# ── FIG 11: Combined country bubble chart ───────────────────────────────────
snap = merged[merged['YEAR']==2024].copy()
snap['youth_u'] = snap['ISO'].map(
    u_1524[u_1524['YEAR']==2024].groupby('ISO')['URATE'].mean())
snap = snap.dropna(subset=['ERATE','URATE','youth_u'])

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(snap['ERATE'], snap['URATE'],
                     s=snap['youth_u']*15,
                     c=snap['youth_u'], cmap='RdYlGn_r',
                     alpha=0.75, edgecolors='white', linewidth=0.5,
                     vmin=5, vmax=35)
for _, row in snap.iterrows():
    ax.annotate(row['ISO'], (row['ERATE'], row['URATE']),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Youth Unemployment Rate (%)", fontsize=10)
ax.set_xlabel("Employment Rate % (15-64)"); ax.set_ylabel("Unemployment Rate % (15-64)")
ax.set_title("Employment vs Unemployment — 2024\nBubble size & colour = Youth (15-24) Unemployment Rate", fontweight='bold', fontsize=13)
plt.tight_layout()

output_path = os.path.join(OUT_DIR,"fig11_bubble_chart.png")
plt.savefig(output_path, dpi=150, bbox_inches='tight')

plt.close()
print(f"  ✓ Saved at {output_path}")

  ✓ Saved at D:\Data Analytics\Project 5\Project 5.2\Output\fig11_bubble_chart.png
